
# Lab 01 - Sentiment Analysis Using Traditional Machine Learning





In [3]:
#Download and load the dataset.
%pip install pandas
%pip install nltk

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

In [5]:
dataset_path = "../data/"

amazon = pd.read_csv(dataset_path + "amazon_cells_labelled.txt", sep="\t", names=["review", "sentiment"], quoting=3)
imdb = pd.read_csv(dataset_path + "imdb_labelled.txt", sep="\t", names=["review", "sentiment"], quoting=3)
yelp = pd.read_csv(dataset_path + "yelp_labelled.txt", sep="\t", names=["review", "sentiment"], quoting=3)

In [6]:
print("Amazon dataset size:", amazon.shape)
print("IMDb dataset:", imdb.shape)
print("Yelp dataset:", yelp.shape)

Amazon dataset size: (1000, 2)
IMDb dataset: (1000, 2)
Yelp dataset: (1000, 2)


In [7]:
# information check of IMDb dataset
print("IMDb dataset information : ")
print("Number of reviews:", len(imdb))

print("\nFirst 5 reviews:")
print(imdb.head())

print("\nLast 5 reviews:")
print(imdb.tail())

print("\nSentiment distribution:")
print(imdb["sentiment"].value_counts())

IMDb dataset information : 
Number of reviews: 1000

First 5 reviews:
                                              review  sentiment
0  A very, very, very slow-moving, aimless movie ...          0
1  Not sure who was more lost - the flat characte...          0
2  Attempting artiness with black & white and cle...          0
3       Very little music or anything to speak of.            0
4  The best scene in the movie was when Gerardo i...          1

Last 5 reviews:
                                                review  sentiment
995  I just got bored watching Jessice Lange take h...          0
996  Unfortunately, any virtue in this film's produ...          0
997                   In a word, it is embarrassing.            0
998                               Exceptionally bad!            0
999  All in all its an insult to one's intelligence...          0

Sentiment distribution:
sentiment
0    500
1    500
Name: count, dtype: int64


In [8]:
data = pd.concat([amazon, imdb, yelp], ignore_index=True)
data.head()

,review,sentiment
0,So there is no way for me to plug it in here i...,0
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
3,Tied to charger for conversations lasting more...,0
4,The mic is great.,1


In [9]:
print("Total reviews:", len(data))
print("Dataset shape:", data.shape)

Total reviews: 3000
Dataset shape: (3000, 2)


In [10]:
# basic text preprocessing
# Convert text to lowercase

data["clean_text"] = data["review"].str.lower()
data[["review", "clean_text"]].head()

,review,clean_text
0,So there is no way for me to plug it in here i...,so there is no way for me to plug it in here i...
1,"Good case, Excellent value.","good case, excellent value."
2,Great for the jawbone.,great for the jawbone.
3,Tied to charger for conversations lasting more...,tied to charger for conversations lasting more...
4,The mic is great.,the mic is great.


In [11]:
#Remove unnecessary characters and punctuation
data["clean_text"] = data["clean_text"].apply(lambda text: re.sub(r"[^a-zA-Z0-9\s]", " ", text))
data[["review", "clean_text"]].head()


,review,clean_text
0,So there is no way for me to plug it in here i...,so there is no way for me to plug it in here i...
1,"Good case, Excellent value.",good case excellent value
2,Great for the jawbone.,great for the jawbone
3,Tied to charger for conversations lasting more...,tied to charger for conversations lasting more...
4,The mic is great.,the mic is great


In [12]:
# Remove extra whitespace
def clean_spaces(text):return " ".join(text.split())
data["clean_text"] = data["clean_text"].apply(clean_spaces)

In [13]:
# Compare original and preprocessed text
data[["review", "clean_text"]].head(10)

,review,clean_text
0,So there is no way for me to plug it in here i...,so there is no way for me to plug it in here i...
1,"Good case, Excellent value.",good case excellent value
2,Great for the jawbone.,great for the jawbone
3,Tied to charger for conversations lasting more...,tied to charger for conversations lasting more...
4,The mic is great.,the mic is great
5,I have to jiggle the plug to get it to line up...,i have to jiggle the plug to get it to line up...
6,If you have several dozen or several hundred c...,if you have several dozen or several hundred c...
7,If you are Razr owner...you must have this!,if you are razr owner you must have this
8,"Needless to say, I wasted my money.",needless to say i wasted my money
9,What a waste of money and time!.,what a waste of money and time


In [14]:
#Explore the text data
#Display positive and negative examples
positive_reviews = data[data["sentiment"] == 1]
positive_reviews[["review", "sentiment"]].head()



,review,sentiment
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
4,The mic is great.,1
7,If you are Razr owner...you must have this!,1
10,And the sound quality is great.,1


In [15]:
negative_reviews = data[data["sentiment"] == 0]
negative_reviews[["review", "sentiment"]].head()

,review,sentiment
0,So there is no way for me to plug it in here i...,0
3,Tied to charger for conversations lasting more...,0
5,I have to jiggle the plug to get it to line up...,0
6,If you have several dozen or several hundred c...,0
8,"Needless to say, I wasted my money.",0


In [16]:
#Identify frequently occurring words
all_words = " ".join(data["clean_text"]).split()
word_counts = Counter(all_words)
common_words = word_counts.most_common(10)
print (common_words)


[('the', 1953), ('and', 1138), ('i', 1037), ('a', 889), ('it', 789), ('is', 754), ('to', 670), ('this', 643), ('of', 624), ('was', 571)]


In [17]:
# compare differences between positive and negative reviews
positive_words = " ".join(positive_reviews["clean_text"]).split()
negative_words = " ".join(negative_reviews["clean_text"]).split()

positive_counts = Counter(positive_words)
negative_counts = Counter(negative_words)

print("Most common words in positive reviews:")
print(positive_counts.most_common(10))

print("\nMost common words in negative reviews:")
print(negative_counts.most_common(10))


Most common words in positive reviews:
[('the', 996), ('and', 674), ('i', 483), ('a', 468), ('is', 417), ('it', 382), ('this', 328), ('of', 310), ('to', 307), ('was', 257)]

Most common words in negative reviews:
[('the', 957), ('i', 554), ('and', 464), ('a', 421), ('it', 407), ('to', 363), ('is', 337), ('this', 315), ('of', 314), ('was', 314)]


In [18]:
#devide the text and sentiment labels
X = data["clean_text"]
y = data["sentiment"]
print("Review data:")
print(X.head())

print("\nSentiment labels:")
print(y.head())

Review data:
0    so there is no way for me to plug it in here i...
1                            good case excellent value
2                                great for the jawbone
3    tied to charger for conversations lasting more...
4                                     the mic is great
Name: clean_text, dtype: str

Sentiment labels:
0    0
1    1
2    1
3    0
4    1
Name: sentiment, dtype: int64


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training data:", len(X_train))
print("Testing data:", len(X_test))

Training data: 2400
Testing data: 600


In [20]:
#Convert Text into Bag of Words (BOW)
#Convert reviews into numerical data
bow_convertor = CountVectorizer()

X_train_bow = bow_convertor.fit_transform(X_train)
X_test_bow = bow_convertor.transform(X_test)

print("Training data shape:", X_train_bow.shape)
print("Testing data shape:", X_test_bow.shape)

Training data shape: (2400, 4512)
Testing data shape: (600, 4512)


In [21]:
print("First review:")
print(X_train.iloc[0])

print("\nNumerical representation:")
print(X_train_bow[0].toarray())

First review:
obviously they have a terrible customer service so you get what you pay for

Numerical representation:
[[0 0 0 ... 0 0 0]]


In [22]:
#Train a ML model 
model = LogisticRegression(max_iter=1000)
model.fit(X_train_bow, y_train)
print("Model training completed succesfully!")


Model training completed succesfully!


In [23]:
#Evaluate the model
predictions = model.predict(X_test_bow)

print("Actual sentiments:")
print(y_test.head(10).tolist())

print("\nPredicted sentiments:")
print(predictions[:10])

Actual sentiments:
[0, 1, 1, 1, 0, 0, 1, 1, 1, 1]

Predicted sentiments:
[0 0 1 1 0 0 1 1 0 1]


In [24]:
#Calculate evaluation scores
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [25]:
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1_score_result = f1_score(y_test, predictions)

print("Model Performance")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1_score_result)

Model Performance
Accuracy: 0.8233333333333334
Precision: 0.8260869565217391
Recall: 0.8205980066445183
F1-score: 0.8233333333333334


In [26]:
# Test the model with a new review

new_review = ["I am happy with this product and it is really good"]

# Clean the review
clean_review = new_review[0].lower()
clean_review = re.sub(r"[^a-zA-Z0-9\s]", " ", clean_review)
clean_review = clean_spaces(clean_review)

# Convert the review into numbers
review_bow = bow_convertor.transform([clean_review])

result = model.predict(review_bow)

if result[0] == 1:
    print("Sentiment: Positive")
else:
    print("Sentiment: Negative")

Sentiment: Positive
